# **Notebook 1: Dual-View Chest X-ray Baseline**

**Description:** This notebook builds the foundation for the X-ray fusion project by converting IU chest X-ray image files and report metadata into a study-level classification dataset. It keeps studies with usable frontal and lateral images, derives the compact five-class target label, creates fixed train/validation/test splits, and trains the first text-only, image-only, and multimodal baselines.

**Main Questions This Notebook Should Answer:**
- Do image features, report text, or simple image/text fusion provide the strongest starting point?
- How much does the normal-class imbalance affect the first results?
- Should the project move toward stronger text encoders and more deliberate fusion designs?

---

# **Part 1: Configuration**
### **Imports**


In [ ]:
from pathlib import Path
import sys
from collections import Counter
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Ensure Python can find the local xray_fusion package.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "src" / "xray_fusion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from xray_fusion.config import DEFAULT_IMAGE_SIZE, DEFAULT_MAX_LEN, find_project_root
from xray_fusion import data as xray_data
from xray_fusion import models as xray_models
from xray_fusion import training as xray_training
from xray_fusion import evaluation as xray_eval
from xray_fusion import segmentation as xray_segmentation
from xray_fusion import gradcam as xray_gradcam


### **Paths**


In [ ]:
proj_path = PROJECT_ROOT / "indiana_projections.csv"
report_path = PROJECT_ROOT / "indiana_reports.csv"
image_dir = PROJECT_ROOT / "images_normalized"
split_dir = PROJECT_ROOT / "splits"
split_dir.mkdir(parents=True, exist_ok=True)


### **Configurations**


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-4


### **Load raw tables**


In [ ]:
proj_df = pd.read_csv(proj_path)
report_df = pd.read_csv(report_path)

print("Projection table shape:", proj_df.shape)
print("Report table shape:", report_df.shape)

display(proj_df.head())
display(report_df.head())


---
# **Part 2: Dataset Preparation**

### **Data Cleaning**


In [ ]:
study_df = xray_data.build_study_level_table(proj_df, report_df, image_dir)

print("Study-level dataframe shape:", study_df.shape)
display(study_df[["uid", "frontal_path", "lateral_path", "indication", "impression"]].head())


### **Inspect report-derived label candidates**


In [ ]:
mesh_counter = Counter()
problem_counter = Counter()

for text in study_df["MeSH"]:
    mesh_counter.update(xray_data.split_semicolon_terms(text))

for text in study_df["Problems"]:
    problem_counter.update(xray_data.split_semicolon_terms(text))

mesh_counts_df = pd.DataFrame(mesh_counter.items(), columns=["label", "count"]).sort_values(
    "count", ascending=False
).reset_index(drop=True)

problem_counts_df = pd.DataFrame(problem_counter.items(), columns=["label", "count"]).sort_values(
    "count", ascending=False
).reset_index(drop=True)

print("Top MeSH labels")
display(mesh_counts_df.head(30))

print("Top Problem labels")
display(problem_counts_df.head(30))


### **Create cleaner study-level labels**

The raw report space is too heterogeneous for a first baseline, so the label space is collapsed into five categories plus an other bucket.


In [ ]:
study_df["clean_label"] = study_df.apply(xray_data.assign_clean_label, axis=1)

label_counts = study_df["clean_label"].value_counts().rename_axis("label").reset_index(name="count")
display(label_counts)


### **Filter to the core classes and create train/validation/test splits**


In [ ]:
keep_labels = [
    "normal",
    "cardiomegaly",
    "pleural_effusion",
    "pneumonia_or_opacity",
    "chronic_lung_disease",
]

model_df = study_df.loc[study_df["clean_label"].isin(keep_labels)].copy()

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["clean_label"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["clean_label"],
    random_state=SEED,
)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name} shape:", df.shape)
    print(df["clean_label"].value_counts(normalize=True).round(3))
    print()

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)

print("Saved split CSVs to:", split_dir)


### **Encode Labels**


In [ ]:
label_encoder = LabelEncoder()
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label_enc"] = label_encoder.fit_transform(train_df["clean_label"])
val_df["label_enc"] = label_encoder.transform(val_df["clean_label"])
test_df["label_enc"] = label_encoder.transform(test_df["clean_label"])

print("Label classes:", list(label_encoder.classes_))


### **Image Transforms**


In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


### **Dataset Classes**


In [ ]:
XRayDualImageDataset = xray_data.XRayDualImageDataset

class XRayMultimodalDataset(xray_data.XRayMultimodalDataset):
    def __init__(self, df, transform=None, max_len=40):
        encode_later = lambda text: encode_text(text, max_len=max_len)
        super().__init__(df, image_transform=transform, max_len=max_len, encode_fn=encode_later)


### **Build dataloaders**


In [ ]:
train_ds = XRayDualImageDataset(train_df, transform=image_transform)
val_ds = XRayDualImageDataset(val_df, transform=image_transform)
test_ds = XRayDualImageDataset(test_df, transform=image_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

frontal_batch, lateral_batch, label_batch = next(iter(train_loader))
print("Frontal batch shape:", tuple(frontal_batch.shape))
print("Lateral batch shape:", tuple(lateral_batch.shape))
print("Label batch shape:", tuple(label_batch.shape))


### **Class Weights**


In [ ]:
class_counts = train_df["clean_label"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / class_counts[label] for label in label_encoder.classes_],
    dtype=torch.float32,
    device=device,
)
class_weights

---

# **Part 3: Baselines**

### **Text-Only Baseline**


In [ ]:
X_train_text = train_df["indication"].fillna("")
X_val_text = val_df["indication"].fillna("")
X_test_text = test_df["indication"].fillna("")

y_train = train_df["clean_label"]
y_val = val_df["clean_label"]
y_test = test_df["clean_label"]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=5000,
)

X_train_vec = vectorizer.fit_transform(X_train_text)
X_val_vec = vectorizer.transform(X_val_text)
X_test_vec = vectorizer.transform(X_test_text)

text_clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)

text_clf.fit(X_train_vec, y_train)

val_pred = text_clf.predict(X_val_vec)
test_pred = text_clf.predict(X_test_vec)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, digits=4))


### **Text-Only Baseline Results Interpretation**

The text-only baseline performed poorly overall, reaching 0.396 test accuracy and 0.242 macro F1. Its balanced class weighting helped it predict some abnormal labels, but precision was low across the disease classes and normal recall dropped to 0.444. This suggests that the indication text alone is too noisy and incomplete to serve as the main diagnostic signal for this dataset.


### **Dual-View Image Baseline Class**


In [ ]:
DualImageModel = xray_models.DualImageClassifier
model = DualImageModel(num_classes=len(label_encoder.classes_)).to(device)


### **Training Functions**


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_epoch = lambda model, loader: xray_training.train_dual_image_epoch(
    model, loader, criterion, optimizer, device, desc="Train"
)
evaluate = lambda model, loader: xray_training.evaluate_dual_image(
    model, loader, criterion, device, desc="Eval"
)


### **Train Model**


In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    print(
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )
    print()


### **Plot training history**


In [ ]:
history_df = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
ax.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Dual-View Image Model Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
ax.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Dual-View Image Model Loss")
ax.legend()
plt.tight_layout()
plt.show()


### **Confusion Matrix**


In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))
print()
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, digits=4))

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(7, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format=".2f", colorbar=False)
ax.set_title("Dual-View Image Model Confusion Matrix")
plt.tight_layout()
plt.show()


### **Dual-View Image Baseline Results Interpretation**

The dual-view image baseline was a much stronger starting point, reaching 0.750 test accuracy compared with 0.396 for text-only. The model learned the normal class well, with 0.994 recall, but it missed most abnormal studies, with disease-class recalls between 0.053 and 0.118. Therefore, the high accuracy is likely mainly driven by the normal-heavy test set instead of balanced disease recognition.


---

# **Part 4: Multimodal Fusion Model**


### **Text Vocabulary + Encoding**


In [ ]:
MAX_VOCAB = 5000
MAX_LEN = 40
vocab = xray_data.build_vocabulary(train_df["indication"].fillna(""), max_vocab=MAX_VOCAB)

def tokenize(text):
    return str(text).lower().split()

def encode_text(text, max_len=MAX_LEN):
    return xray_data.encode_text(text, vocab, max_len)

train_mm_ds = XRayMultimodalDataset(train_df, transform=image_transform, max_len=MAX_LEN)
val_mm_ds = XRayMultimodalDataset(val_df, transform=image_transform, max_len=MAX_LEN)
test_mm_ds = XRayMultimodalDataset(test_df, transform=image_transform, max_len=MAX_LEN)

train_mm_loader = DataLoader(train_mm_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_mm_loader = DataLoader(val_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_mm_loader = DataLoader(test_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

frontal_batch, lateral_batch, text_batch, label_batch = next(iter(train_mm_loader))
print("Fusion frontal batch shape:", tuple(frontal_batch.shape))
print("Fusion lateral batch shape:", tuple(lateral_batch.shape))
print("Fusion text batch shape:", tuple(text_batch.shape))
print("Fusion label batch shape:", tuple(label_batch.shape))


### **Model Class**


In [ ]:
MultimodalFusionModel = xray_models.SimpleVocabularyFusionClassifier

fusion_model = MultimodalFusionModel(
    num_classes=len(label_encoder.classes_),
    vocab_size=len(vocab),
).to(device)

fusion_optimizer = torch.optim.Adam(fusion_model.parameters(), lr=LR)
fusion_criterion = nn.CrossEntropyLoss(weight=class_weights)


### **Training Functions**


In [ ]:
train_epoch_fusion = lambda model, loader: xray_training.train_simple_fusion_epoch(
    model, loader, fusion_criterion, fusion_optimizer, device, desc="Train Fusion"
)
evaluate_fusion = lambda model, loader: xray_training.evaluate_simple_fusion(
    model, loader, fusion_criterion, device, desc="Eval Fusion"
)


### **Train Model**


In [ ]:
fusion_history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_epoch_fusion(fusion_model, train_mm_loader)
    val_loss, val_acc, _, _ = evaluate_fusion(fusion_model, val_mm_loader)

    fusion_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    print(
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )
    print()

### **Confusion Matrix**


In [ ]:
fusion_test_loss, fusion_test_acc, y_true, y_pred = evaluate_fusion(fusion_model, test_mm_loader)

print("Fusion Test Loss:", round(fusion_test_loss, 4))
print("Fusion Test Accuracy:", round(fusion_test_acc, 4))
print()
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, digits=4))

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(7, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format=".2f", colorbar=False)
ax.set_title("Multimodal Fusion Confusion Matrix")
plt.tight_layout()
plt.show()

### **Multimodal Fusion Model Results Interpretation**

The first multimodal fusion model produced the best overall accuracy at 0.769 and improved macro F1 to 0.368. It also raised abnormal-class recall compared with the dual-view image baseline for cardiomegaly, chronic lung disease, pleural effusion, and pneumonia/opacity. Even so, normal recall reached 1.000 while abnormal recall remained low, so this simple fusion setup still does not solve the class-imbalance problem.



### **Final Model Comparison**


In [ ]:
comparison_df = pd.DataFrame([
    {"model": "text_only", "test_acc": accuracy_score(y_test, test_pred)},
    {"model": "dual_view_image", "test_acc": test_acc},
    {"model": "fusion", "test_acc": fusion_test_acc},
]).sort_values("test_acc", ascending=False)

display(comparison_df)

### **Final Model Comparison Interpretation**

The final comparison shows that the text-only model performs the worst, with an accuracy of 0.396, while the dual-view image model performs better at 0.750. The simple image-text fusion model achieves the highest accuracy at 0.769. These results demonstrate that image data provides a much stronger basis for prediction than using text data alone. Combining image and text information also improves results, although the gain over the image-only model is somewhat small. This suggests that multimodal medical imaging systems have the potential to outperform models that rely exclusively on either images or text. Later experiments will retain the fusion model and investigate whether additional improvements can be achieved through more advanced multimodal approaches.



---

# **Part 5: Conclusion**

This notebook established the core dataset and baseline modeling problem for the project. It converted the IU chest X-ray image/report metadata into a five-class study-level classification task, retained studies with both frontal and lateral images, and created fixed train, validation, and test splits that will be reused throughout later notebooks. This provides a consistent foundation for comparing future experiments using the same label space, paired-image requirement, and data splits.

The baseline results show that image information provides the strongest starting point. The text-only model reached 0.396 test accuracy, while the dual-view image model improved substantially to 0.750, showing that the frontal and lateral X-ray views contain much stronger predictive signal than the initial text representation. The first multimodal fusion model performed best overall, reaching 0.769 test accuracy and a macro F1 of 0.368. This supports the main direction of the project, showing that combining image and text information can improve performance beyond either modality alone. However, the improvement over image-only classification is relatively small, and performance remains heavily influenced by the normal class, with much weaker recall across the abnormal categories.

These results support continuing with multimodal learning while also showing the class-imbalance and minority-class recall problems that later notebooks need to address. More broadly, the results suggest that when both imaging and relevant text data are available, combining the two may provide more useful context. For example, in a clinical system, information from someones medical record could be incorporated with appropriate consent and privacy protections, allowing prior clinical context to complement the image data instead of relying only on the image.

---
